# Construction Phase Detection — real training pipeline

This notebook trains the phase detector on the generated construction dataset:

- `activities_with_equipment.csv` — plan, phases, activities and expected equipment
- `equipment_events_with_context.csv` — synthetic video-tracker arrival/departure events
- `equipment_descriptions.csv` — semantic descriptions of the 13 equipment classes

The training data is built as daily temporal observations from the event stream, with realistic detector misses/false positives. Projects are split using the dataset's `split` field, so no project is shared between train/validation/test.


In [1]:
# 1. Google Drive + paths

from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/data")
ACTIVITIES_PATH = DATA_DIR / "activities_with_equipment.csv"
EVENTS_PATH = DATA_DIR / "equipment_events_with_context.csv"
EQUIPMENT_DESC_PATH = DATA_DIR / "equipment_descriptions.csv"

CHECKPOINT_DIR = DATA_DIR / "construction_phase_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Activities:", ACTIVITIES_PATH)
print("Events:", EVENTS_PATH)
print("Checkpoints:", CHECKPOINT_DIR)


Mounted at /content/drive
Activities: /content/drive/MyDrive/Colab Notebooks/data/activities_with_equipment.csv
Events: /content/drive/MyDrive/Colab Notebooks/data/equipment_events_with_context.csv
Checkpoints: /content/drive/MyDrive/Colab Notebooks/data/construction_phase_checkpoints


In [ ]:
# 1. Local paths

from pathlib import Path

# Текущая рабочая директория
PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data"

ACTIVITIES_PATH = DATA_DIR / "activities_with_equipment.csv"
EVENTS_PATH = DATA_DIR / "equipment_events_with_context.csv"
EQUIPMENT_DESC_PATH = DATA_DIR / "equipment_descriptions.csv"

CHECKPOINT_DIR = PROJECT_DIR / "construction_phase_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Activities:", ACTIVITIES_PATH)
print("Events:", EVENTS_PATH)
print("Equipment descriptions:", EQUIPMENT_DESC_PATH)
print("Checkpoints:", CHECKPOINT_DIR)

# Проверяем, что все необходимые файлы существуют
required_files = [
    ACTIVITIES_PATH,
    EVENTS_PATH,
    EQUIPMENT_DESC_PATH,
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "Не найдены следующие файлы:\n"
        + "\n".join(str(path) for path in missing_files)
    )

In [2]:
# 2. Imports, reproducibility, device

import copy
import math
import random
from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


## 3. Load and inspect the real training source

In [3]:
activities = pd.read_csv(ACTIVITIES_PATH)
events = pd.read_csv(EVENTS_PATH)
equipment_desc = pd.read_csv(EQUIPMENT_DESC_PATH)

activities["expected_equipment"] = activities["expected_equipment"].apply(
    lambda x: eval(x) if isinstance(x, str) else x
)

events["at"] = pd.to_datetime(events["at"])

print("activities:", activities.shape)
print("events:", events.shape)
print("projects:", activities["project_id"].nunique())
print("phases:", activities["phase"].nunique())
print("equipment classes:", equipment_desc["equipment_class"].nunique())

print("\nProject split:")
print(activities.groupby("split")["project_id"].nunique())

print("\nPhases:")
print(
    activities[["phase", "phase_order"]]
    .drop_duplicates()
    .sort_values("phase_order")
    .to_string(index=False)
)


activities: (9279, 21)
events: (32205, 8)
projects: 100
phases: 10
equipment classes: 13

Project split:
split
test          15
train         70
validation    15
Name: project_id, dtype: int64

Phases:
           phase  phase_order
 Preconstruction            1
Site Preparation            2
       Earthwork            3
      Foundation            4
Structural Frame            5
         Masonry            6
             MEP            7
       Finishing            8
  External Works            9
   Commissioning           10


## 4. Canonical equipment representation

Each temporal observation contains three values per equipment class:

1. presence/count estimate,
2. detector visibility,
3. detector confidence.

This is deliberately different from treating a missing detection as a true departure.


In [4]:
EQUIPMENT_CLASSES = equipment_desc["equipment_class"].tolist()
EQUIPMENT_TO_ID = {x: i for i, x in enumerate(EQUIPMENT_CLASSES)}
NUM_EQUIPMENT = len(EQUIPMENT_CLASSES)

equipment_descriptions = dict(
    zip(equipment_desc["equipment_class"], equipment_desc["description"])
)

print(EQUIPMENT_CLASSES)


['worker', 'tower_crane', 'hanging_hook', 'vehicle_crane', 'roller', 'bulldozer', 'excavator', 'truck', 'loader', 'pump_truck', 'concrete_mixer', 'pile_driver', 'other_vehicle']


## 5. Phase metadata and semantic phase embeddings

Phase text is generated from the actual plan:

- phase name,
- activities belonging to the phase,
- expected equipment,
- equipment descriptions.

The text encoder is pretrained; its embeddings are then consumed by the trainable `SemanticPhaseEncoder`.


In [5]:
# Phase metadata
phase_meta = (
    activities.groupby(["phase", "phase_order"], as_index=False)
    .agg(
        activities=("activity_name", lambda x: list(dict.fromkeys(x))),
        duration_days=("planned_duration_days", "sum"),
        criticality=("criticality", "mean"),
    )
    .sort_values("phase_order")
    .reset_index(drop=True)
)

PHASE_NAMES = phase_meta["phase"].tolist()
PHASE_TO_ID = {p: i for i, p in enumerate(PHASE_NAMES)}
NUM_PHASES = len(PHASE_NAMES)

phase_equipment = {}
phase_texts = []

for _, row in phase_meta.iterrows():
    phase = row["phase"]
    phase_activities = activities.loc[
        activities["phase"] == phase, "expected_equipment"
    ]

    eq_set = set()
    for eqs in phase_activities:
        eq_set.update(eqs)
    eqs = [e for e in EQUIPMENT_CLASSES if e in eq_set]
    phase_equipment[phase] = eqs

    equipment_text = "\n".join(
        f"- {e}: {equipment_descriptions[e]}" for e in eqs
    )
    activities_text = ", ".join(row["activities"])

    phase_texts.append(
        f"Construction phase: {phase}. "
        f"Activities: {activities_text}. "
        f"Expected equipment: {', '.join(eqs)}.\n"
        f"Equipment descriptions:\n{equipment_text}"
    )

phase_meta["phase_text"] = phase_texts

print(phase_meta[["phase", "duration_days", "criticality"]].to_string(index=False))


           phase  duration_days  criticality
 Preconstruction           5293     0.552343
Site Preparation           3919     0.552279
       Earthwork          11438     0.551491
      Foundation          11302     0.554317
Structural Frame          30364     0.548564
         Masonry          24536     0.543307
             MEP          21222     0.551121
       Finishing          25112     0.551413
  External Works           6276     0.558091
   Commissioning           3051     0.556084


In [6]:
# Install/load a pretrained sentence encoder in Colab.
# The model is used only to create semantic phase embeddings; the main model is trained below.

!pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer

TEXT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
text_encoder = SentenceTransformer(TEXT_MODEL_NAME, device=str(device))

with torch.no_grad():
    phase_text_embeddings = text_encoder.encode(
        phase_meta["phase_text"].tolist(),
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).float().cpu()

PHASE_TEXT_DIM = phase_text_embeddings.shape[-1]

print("Phase text embedding shape:", phase_text_embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Phase text embedding shape: torch.Size([10, 384])


## 6. Convert tracker events into temporal video-like observations

The source events are `arrival/departure`. We reconstruct the equipment state at a regular temporal grid.

Noise is added only to the observed equipment features:

- random missed detections,
- confidence variation,
- occasional false positives.

Ground-truth phase labels still come from the plan, not from the noisy detector.


In [7]:
OBSERVATION_STEP_HOURS = 24
WINDOW_SIZE = 128
WINDOW_STRIDE = 64

MISS_PROB = 0.08
FALSE_POSITIVE_PROB = 0.01
CONFIDENCE_NOISE = 0.08

def build_project_timeline(project_id: str):
    plan = (
        activities[activities["project_id"] == project_id]
        .sort_values("activity_sequence")
        .copy()
    )
    ev = events[events["project_id"] == project_id].sort_values("at").copy()

    start = ev["at"].min().floor("D")
    end = ev["at"].max().ceil("D")

    timestamps = pd.date_range(
        start=start + pd.Timedelta(hours=12),
        end=end,
        freq=f"{OBSERVATION_STEP_HOURS}h",
    )

    # Event-state reconstruction.
    state = {e: 0 for e in EQUIPMENT_CLASSES}
    event_idx = 0
    ev_records = ev[["equipment_class", "event", "at"]].to_dict("records")

    observations = []
    labels = []

    # Build activity intervals from the plan. This matches the synthetic
    # event generation used for the current dataset.
    activity_start = start
    intervals = []
    for _, r in plan.iterrows():
        duration = max(float(r["planned_duration_days"]), 1.0)
        activity_end = activity_start + pd.Timedelta(days=duration)
        intervals.append(
            (activity_start, activity_end, PHASE_TO_ID[r["phase"]])
        )
        activity_start = activity_end

    interval_idx = 0

    for ts in timestamps:
        while event_idx < len(ev_records) and pd.Timestamp(ev_records[event_idx]["at"]) <= ts:
            e = ev_records[event_idx]
            if e["event"] == "arrival":
                state[e["equipment_class"]] += 1
            else:
                state[e["equipment_class"]] = max(
                    0, state[e["equipment_class"]] - 1
                )
            event_idx += 1

        while interval_idx < len(intervals) - 1 and ts >= intervals[interval_idx][1]:
            interval_idx += 1

        if not intervals:
            continue

        start_i, end_i, phase_id = intervals[interval_idx]
        if start_i <= ts < end_i:
            labels.append(phase_id)
        else:
            # Outside plan coverage: use nearest valid interval endpoint.
            labels.append(
                intervals[max(0, min(interval_idx, len(intervals)-1))][2]
            )

        counts = np.array(
            [state[e] for e in EQUIPMENT_CLASSES],
            dtype=np.float32,
        )

        # Detector-like noisy observation.
        visible = np.ones(NUM_EQUIPMENT, dtype=np.float32)
        confidence = np.ones(NUM_EQUIPMENT, dtype=np.float32)

        for i in range(NUM_EQUIPMENT):
            if counts[i] > 0 and random.random() < MISS_PROB:
                visible[i] = 0.0
                confidence[i] = 0.0
            elif counts[i] > 0:
                confidence[i] = np.clip(
                    np.random.normal(0.9, CONFIDENCE_NOISE), 0.35, 1.0
                )

            # Occasional false positive.
            if counts[i] == 0 and random.random() < FALSE_POSITIVE_PROB:
                visible[i] = 1.0
                confidence[i] = np.clip(
                    np.random.normal(0.55, CONFIDENCE_NOISE), 0.2, 0.9
                )

        # Normalize counts so the feature scale is stable.
        count_feature = np.clip(counts, 0, 5) / 5.0

        obs = np.concatenate([
            count_feature,
            visible,
            confidence,
        ]).astype(np.float32)

        observations.append(obs)

    return (
        np.asarray(observations, dtype=np.float32),
        np.asarray(labels, dtype=np.int64),
        timestamps[:len(labels)],
    )


# Quick structural check — this is not a smoke training run.
x0, y0, t0 = build_project_timeline(activities["project_id"].iloc[0])
print("One project observations:", x0.shape)
print("One project labels:", y0.shape)
print("First timestamps:", t0[:3].tolist())


One project observations: (1664, 39)
One project labels: (1664,)
First timestamps: [Timestamp('2024-01-01 12:00:00'), Timestamp('2024-01-02 12:00:00'), Timestamp('2024-01-03 12:00:00')]


## 7. Windowed project dataset

The Transformer sees windows of `WINDOW_SIZE` observations. Splitting is done at project level before creating windows.

A window may contain one phase or several consecutive phases, which is important for learning temporal transitions.


In [8]:
class ConstructionWindowDataset(Dataset):
    def __init__(self, project_ids):
        self.samples = []

        for project_id in project_ids:
            observations, labels, timestamps = build_project_timeline(project_id)

            if len(labels) == 0:
                continue

            if len(labels) <= WINDOW_SIZE:
                self.samples.append(
                    (observations, labels, timestamps, project_id)
                )
                continue

            for start in range(0, len(labels) - WINDOW_SIZE + 1, WINDOW_STRIDE):
                end = start + WINDOW_SIZE
                self.samples.append(
                    (
                        observations[start:end],
                        labels[start:end],
                        timestamps[start:end],
                        project_id,
                    )
                )

            # Always retain the final window.
            if (len(labels) - WINDOW_SIZE) % WINDOW_STRIDE != 0:
                start = len(labels) - WINDOW_SIZE
                self.samples.append(
                    (
                        observations[start:],
                        labels[start:],
                        timestamps[start:],
                        project_id,
                    )
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        observations, labels, timestamps, project_id = self.samples[idx]

        return {
            "observations": torch.from_numpy(observations),
            "labels": torch.from_numpy(labels),
            "timestamps": list(timestamps),
            "project_id": project_id,
        }


all_projects = (
    activities[["project_id", "split"]]
    .drop_duplicates()
)

train_projects = all_projects.loc[
    all_projects["split"] == "train", "project_id"
].tolist()
val_projects = all_projects.loc[
    all_projects["split"] == "validation", "project_id"
].tolist()
test_projects = all_projects.loc[
    all_projects["split"] == "test", "project_id"
].tolist()

train_dataset = ConstructionWindowDataset(train_projects)
val_dataset = ConstructionWindowDataset(val_projects)
test_dataset = ConstructionWindowDataset(test_projects)

print("Train projects:", len(train_projects), "windows:", len(train_dataset))
print("Val projects:", len(val_projects), "windows:", len(val_dataset))
print("Test projects:", len(test_projects), "windows:", len(test_dataset))


Train projects: 70 windows: 1540
Val projects: 15 windows: 314
Test projects: 15 windows: 325


In [9]:
# Phase metadata is shared across all samples, so it does not belong inside
# the Dataset itself. We broadcast it to each batch here.

def collate_batch(batch):
    observations = torch.stack([x["observations"] for x in batch])
    labels = torch.stack([x["labels"] for x in batch])

    B = observations.shape[0]

    phase_text = phase_text_embeddings.unsqueeze(0).expand(
        B, -1, -1
    ).clone()

    # Structured phase features:
    # expected equipment (13), normalized duration, mean criticality.
    structured = []

    duration = phase_meta["duration_days"].to_numpy(dtype=np.float32)
    duration = duration / max(duration.max(), 1.0)

    criticality = phase_meta["criticality"].to_numpy(dtype=np.float32)

    for phase in PHASE_NAMES:
        eq = np.zeros(NUM_EQUIPMENT, dtype=np.float32)
        for e in phase_equipment[phase]:
            eq[EQUIPMENT_TO_ID[e]] = 1.0

        pidx = PHASE_TO_ID[phase]
        structured.append(
            np.concatenate([
                eq,
                np.array([duration[pidx], criticality[pidx]], dtype=np.float32)
            ])
        )

    phase_structured = torch.tensor(
        np.stack(structured),
        dtype=torch.float32
    ).unsqueeze(0).expand(B, -1, -1).clone()

    return {
        "observations": observations,
        "labels": labels,
        "phase_text_embeddings": phase_text,
        "phase_structured": phase_structured,
        "timestamps": [x["timestamps"] for x in batch],
        "project_id": [x["project_id"] for x in batch],
    }


BATCH_SIZE = 8
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    collate_fn=collate_batch,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    collate_fn=collate_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    collate_fn=collate_batch,
)

batch = next(iter(train_loader))
print("Observation batch:", batch["observations"].shape)
print("Labels batch:", batch["labels"].shape)
print("Phase text:", batch["phase_text_embeddings"].shape)
print("Phase structured:", batch["phase_structured"].shape)


Observation batch: torch.Size([8, 128, 39])
Labels batch: torch.Size([8, 128])
Phase text: torch.Size([8, 10, 384])
Phase structured: torch.Size([8, 10, 15])


## 8. Model

In [10]:
OBSERVATION_DIM = NUM_EQUIPMENT * 3
PHASE_STRUCTURED_DIM = NUM_EQUIPMENT + 2
PHASE_TEXT_DIM = int(PHASE_TEXT_DIM)

@dataclass
class PhaseModelOutput:
    emissions: torch.Tensor
    phase_embeddings: torch.Tensor
    temporal_embeddings: torch.Tensor


class ObservationEncoder(nn.Module):
    def __init__(self, observation_dim, d_model=256, dropout=0.1):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(observation_dim, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

    def forward(self, x):
        return self.network(x)


class SemanticPhaseEncoder(nn.Module):
    def __init__(self, text_embedding_dim, structured_dim, d_model=256, dropout=0.1):
        super().__init__()
        self.text_projection = nn.Linear(text_embedding_dim, d_model)
        self.structured_projection = nn.Linear(structured_dim, d_model)
        self.fusion = nn.Sequential(
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
        )

    def forward(self, text_embeddings, structured_features):
        text = self.text_projection(text_embeddings)
        structured = self.structured_projection(structured_features)
        return self.fusion(torch.cat([text, structured], dim=-1))


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x):
        if x.size(1) > self.pe.size(1):
            raise ValueError(
                f"Sequence length {x.size(1)} exceeds max positional length "
                f"{self.pe.size(1)}."
            )
        return x + self.pe[:, :x.size(1)].to(dtype=x.dtype)


class TemporalTransformer(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_layers=4,
                 dim_feedforward=1024, dropout=0.1):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, padding_mask=None):
        return self.norm(
            self.encoder(x, src_key_padding_mask=padding_mask)
        )


class PhaseScorer(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.observation_projection = nn.Linear(d_model, d_model, bias=False)
        self.phase_projection = nn.Linear(d_model, d_model, bias=False)
        self.scale = d_model ** -0.5

    def forward(self, temporal, phases):
        q = self.observation_projection(temporal)
        k = self.phase_projection(phases)
        return torch.einsum("btd,bpd->btp", q, k) * self.scale


class ConstructionPhaseModel(nn.Module):
    def __init__(
        self,
        observation_dim,
        phase_text_dim,
        phase_structured_dim,
        d_model=256,
        nhead=8,
        num_layers=4,
        dim_feedforward=1024,
        dropout=0.1,
        max_len=4096,
    ):
        super().__init__()
        self.observation_encoder = ObservationEncoder(
            observation_dim, d_model, dropout
        )
        self.phase_encoder = SemanticPhaseEncoder(
            phase_text_dim, phase_structured_dim, d_model, dropout
        )
        self.position_encoding = SinusoidalPositionalEncoding(
            d_model, max_len
        )
        self.temporal_transformer = TemporalTransformer(
            d_model, nhead, num_layers, dim_feedforward, dropout
        )
        self.phase_scorer = PhaseScorer(d_model)

    def forward(
        self,
        observations,
        phase_text_embeddings,
        phase_structured,
        padding_mask=None,
    ):
        x = self.observation_encoder(observations)
        x = self.position_encoding(x)
        temporal = self.temporal_transformer(x, padding_mask)
        phases = self.phase_encoder(
            phase_text_embeddings,
            phase_structured,
        )
        emissions = self.phase_scorer(temporal, phases)

        return PhaseModelOutput(
            emissions=emissions,
            phase_embeddings=phases,
            temporal_embeddings=temporal,
        )


## 9. Loss, metrics and checkpointing

In [11]:
def phase_classification_loss(emissions, labels):
    B, T, P = emissions.shape
    return F.cross_entropy(
        emissions.reshape(B * T, P),
        labels.reshape(B * T),
    )


def save_checkpoint(
    model,
    optimizer,
    scheduler,
    epoch,
    train_metrics,
    val_metrics,
    best_val_loss,
    path,
):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
            "train_metrics": train_metrics,
            "val_metrics": val_metrics,
            "best_val_loss": best_val_loss,
            "phase_names": PHASE_NAMES,
            "equipment_classes": EQUIPMENT_CLASSES,
            "config": {
                "window_size": WINDOW_SIZE,
                "observation_dim": OBSERVATION_DIM,
                "phase_text_dim": PHASE_TEXT_DIM,
                "phase_structured_dim": PHASE_STRUCTURED_DIM,
            },
        },
        path,
    )


def run_epoch(model, loader, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for batch in loader:
        observations = batch["observations"].to(device, non_blocking=True)
        phase_text = batch["phase_text_embeddings"].to(device, non_blocking=True)
        phase_structured = batch["phase_structured"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            output = model(
                observations=observations,
                phase_text_embeddings=phase_text,
                phase_structured=phase_structured,
            )

            loss = phase_classification_loss(output.emissions, labels)

            if is_train:
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(), 1.0
                    )
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(), 1.0
                    )
                    optimizer.step()

        pred = output.emissions.argmax(dim=-1)

        total_loss += loss.item() * observations.size(0)
        total_correct += (pred == labels).sum().item()
        total_count += labels.numel()

    return {
        "loss": total_loss / max(len(loader.dataset), 1),
        "accuracy": total_correct / max(total_count, 1),
    }


## 10. Real training

In [12]:
# Model configuration
D_MODEL = 256
NHEAD = 8
NUM_LAYERS = 4
DIM_FEEDFORWARD = 1024
DROPOUT = 0.1

EPOCHS = 30
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4

model = ConstructionPhaseModel(
    observation_dim=OBSERVATION_DIM,
    phase_text_dim=PHASE_TEXT_DIM,
    phase_structured_dim=PHASE_STRUCTURED_DIM,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    max_len=WINDOW_SIZE,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda"),
)

print("Parameters:", sum(p.numel() for p in model.parameters()))


Parameters: 3602176


/tmp/ipykernel_3054/1614219393.py:82: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


In [13]:
# Train.
# A checkpoint is written to Google Drive after EVERY epoch.
# best.pt always contains the best validation-loss model.

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scaler=scaler,
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        optimizer=None,
        scaler=None,
    )

    scheduler.step()

    history["train_loss"].append(train_metrics["loss"])
    history["train_accuracy"].append(train_metrics["accuracy"])
    history["val_loss"].append(val_metrics["loss"])
    history["val_accuracy"].append(val_metrics["accuracy"])

    # Every epoch checkpoint
    epoch_path = CHECKPOINT_DIR / f"checkpoint_epoch_{epoch:03d}.pt"
    save_checkpoint(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=epoch,
        train_metrics=train_metrics,
        val_metrics=val_metrics,
        best_val_loss=best_val_loss,
        path=epoch_path,
    )

    # Best checkpoint
    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]

        best_path = CHECKPOINT_DIR / "best.pt"
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            train_metrics=train_metrics,
            val_metrics=val_metrics,
            best_val_loss=best_val_loss,
            path=best_path,
        )
        best_marker = "  <-- BEST"
    else:
        best_marker = ""

    lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"lr={lr:.2e} | "
        f"train loss={train_metrics['loss']:.4f} | "
        f"train acc={train_metrics['accuracy']:.4f} | "
        f"val loss={val_metrics['loss']:.4f} | "
        f"val acc={val_metrics['accuracy']:.4f}"
        f"{best_marker}"
    )

print("Training finished.")
print("Best validation loss:", best_val_loss)
print("Checkpoints:", CHECKPOINT_DIR)


Epoch 001/30 | lr=2.99e-04 | train loss=1.7547 | train acc=0.2922 | val loss=1.9519 | val acc=0.3079  <-- BEST
Epoch 002/30 | lr=2.97e-04 | train loss=0.9919 | train acc=0.5945 | val loss=0.7034 | val acc=0.7328  <-- BEST
Epoch 003/30 | lr=2.93e-04 | train loss=0.6958 | train acc=0.7131 | val loss=0.6397 | val acc=0.7259  <-- BEST
Epoch 004/30 | lr=2.87e-04 | train loss=0.6043 | train acc=0.7463 | val loss=0.5088 | val acc=0.7736  <-- BEST
Epoch 005/30 | lr=2.80e-04 | train loss=0.5563 | train acc=0.7727 | val loss=0.5023 | val acc=0.8075  <-- BEST
Epoch 006/30 | lr=2.71e-04 | train loss=0.4226 | train acc=0.8220 | val loss=0.4398 | val acc=0.8322  <-- BEST
Epoch 007/30 | lr=2.61e-04 | train loss=0.4071 | train acc=0.8210 | val loss=0.5326 | val acc=0.7693
Epoch 008/30 | lr=2.50e-04 | train loss=0.3477 | train acc=0.8574 | val loss=0.4839 | val acc=0.8264
Epoch 009/30 | lr=2.38e-04 | train loss=0.2916 | train acc=0.8870 | val loss=0.3498 | val acc=0.8608  <-- BEST
Epoch 010/30 | lr=2.2

## 11. Load best model and evaluate on held-out test projects

In [14]:
best_path = CHECKPOINT_DIR / "best.pt"

checkpoint = torch.load(
    best_path,
    map_location=device,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

test_metrics = run_epoch(
    model,
    test_loader,
    optimizer=None,
    scaler=None,
)

print("Test loss:", test_metrics["loss"])
print("Test accuracy:", test_metrics["accuracy"])


Test loss: 0.2722554169709866
Test accuracy: 0.8932692307692308


In [15]:
# Per-phase metrics and confusion matrix on the held-out test projects.

from sklearn.metrics import classification_report, confusion_matrix

all_true = []
all_pred = []

with torch.no_grad():
    for batch in test_loader:
        observations = batch["observations"].to(device)
        phase_text = batch["phase_text_embeddings"].to(device)
        phase_structured = batch["phase_structured"].to(device)
        labels = batch["labels"].to(device)

        output = model(
            observations=observations,
            phase_text_embeddings=phase_text,
            phase_structured=phase_structured,
        )

        pred = output.emissions.argmax(dim=-1)

        all_true.append(labels.cpu().numpy().reshape(-1))
        all_pred.append(pred.cpu().numpy().reshape(-1))

all_true = np.concatenate(all_true)
all_pred = np.concatenate(all_pred)

print(
    classification_report(
        all_true,
        all_pred,
        labels=list(range(NUM_PHASES)),
        target_names=PHASE_NAMES,
        zero_division=0,
    )
)

cm = confusion_matrix(
    all_true,
    all_pred,
    labels=list(range(NUM_PHASES)),
)

cm_df = pd.DataFrame(
    cm,
    index=PHASE_NAMES,
    columns=PHASE_NAMES,
)

display(cm_df)


                  precision    recall  f1-score   support

 Preconstruction       0.99      0.93      0.96       836
Site Preparation       0.95      0.99      0.97      1042
       Earthwork       1.00      1.00      1.00      3358
      Foundation       0.98      0.97      0.98      3290
Structural Frame       0.94      0.85      0.89      9294
         Masonry       0.82      0.91      0.86      7256
             MEP       0.88      0.71      0.78      6322
       Finishing       0.81      0.95      0.88      7881
  External Works       1.00      1.00      1.00      1779
   Commissioning       1.00      1.00      1.00       542

        accuracy                           0.89     41600
       macro avg       0.94      0.93      0.93     41600
    weighted avg       0.90      0.89      0.89     41600



,Preconstruction,Site Preparation,Earthwork,Foundation,Structural Frame,Masonry,MEP,Finishing,External Works,Commissioning
Preconstruction,781,55,0,0,0,0,0,0,0,0
Site Preparation,9,1029,4,0,0,0,0,0,0,0
Earthwork,0,0,3358,0,0,0,0,0,0,0
Foundation,0,0,0,3198,86,6,0,0,0,0
Structural Frame,0,0,0,51,7893,1350,0,0,0,0
Masonry,0,0,0,0,409,6593,251,3,0,0
MEP,0,0,0,0,0,111,4482,1729,0,0
Finishing,0,0,0,0,0,0,376,7505,0,0
External Works,0,0,0,0,0,0,0,0,1779,0
Commissioning,0,0,0,0,0,0,0,0,0,542


## 12. Resume training after a Colab disconnect

Run this cell instead of reinitializing the optimizer/model when you want to continue from a saved epoch.


In [ ]:
def resume_from_checkpoint(checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint.get("best_val_loss", float("inf"))

    return start_epoch, best_val_loss


# Example:
# start_epoch, best_val_loss = resume_from_checkpoint(
#     CHECKPOINT_DIR / "checkpoint_epoch_010.pt"
# )
# print("Resume from epoch:", start_epoch)
